In [6]:
from pathlib import Path
import os, sys

repo_root = Path.cwd()
while repo_root.name != "DenseFlow_Tool-main" and repo_root.parent != repo_root:
    repo_root = repo_root.parent
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [7]:
import importlib
import numpy as np
import time
import Code.myHoloscope as myHoloscope
import Code.myMaxflow as myMaxflow
import Code.Check as check
import Code.info as info
import Code.holodatatran as holo
import Code.Cubedatatran as cube
import Code.Datatran_1 as dt
import Code.compare_method as cm
importlib.reload(myHoloscope)
importlib.reload(myMaxflow)
from Code.myHoloscope import *
from Code.myMaxflow import *
from Code.Check import *
from Code.info import *
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(time, 'clock'):
    time.clock = time.perf_counter
caseing = ['PlusTokenPonzi']
caseandk = {'PlusTokenPonzi':10 }

In [8]:
from pathlib import Path
import ast
import pandas as pd

base = Path('./inputData/AML')
data_base = base / 'data'
cases_now = ['AlphaHomora', 'CryptopiaHacker', 'PlusTokenPonzi']

def ensure_normal_tx(case):
    case_dir = base / case
    normal_tx = case_dir / 'all-normal-tx.csv'
    if normal_tx.exists():
        return normal_tx

    all_tx = case_dir / 'all-tx.csv'
    cube_xy = case_dir / 'all-normal-tx-cube_xy.csv'
    addr_csv = case_dir / 'all-address.csv'

    if all_tx.exists():
        print(f'[{case}] building all-normal-tx.csv from all-tx.csv')
        df = pd.read_csv(all_tx)
        if 'isError' not in df.columns:
            df['isError'] = 0
        df.to_csv(normal_tx, index=False)
        return normal_tx

    if cube_xy.exists() and addr_csv.exists():
        print(f'[{case}] building all-normal-tx.csv from cube_xy + all-address')
        cube = pd.read_csv(cube_xy, header=None, names=['from_id', 'to_id', 'timeStamp', 'value'])
        addr = pd.read_csv(addr_csv)
        addresses = addr['address'].astype(str).tolist()
        max_idx = int(max(cube['from_id'].max(), cube['to_id'].max()))
        if max_idx >= len(addresses):
            raise ValueError(f'[{case}] address list too short for cube ids: {max_idx} >= {len(addresses)}')
        cube['from'] = cube['from_id'].astype(int).map(lambda i: addresses[i])
        cube['to'] = cube['to_id'].astype(int).map(lambda i: addresses[i])
        out = pd.DataFrame({
            'hash': [f'{case}_{i}' for i in range(len(cube))],
            'from': cube['from'],
            'to': cube['to'],
            'value': pd.to_numeric(cube['value'], errors='coerce').fillna(0.0),
            'timeStamp': pd.to_numeric(cube['timeStamp'], errors='coerce').fillna(0).astype(int),
            'blockNumber': 0,
            'tokenSymbol': '',
            'contractAddress': '',
            'isError': 0,
            'gasPrice': 0,
            'gasUsed': 0,
        })
        out.to_csv(normal_tx, index=False)
        return normal_tx

    raise FileNotFoundError(f'[{case}] missing source files to build all-normal-tx.csv')

def ensure_addr_maps(case, normal_tx):
    case_data_dir = data_base / case
    case_data_dir.mkdir(parents=True, exist_ok=True)
    from_file = case_data_dir / 'all-normal-tx_from.txt'
    to_file = case_data_dir / 'all-normal-tx_to.txt'

    if from_file.exists() and to_file.exists():
        from_list = ast.literal_eval(from_file.read_text(encoding='utf-8'))
        to_list = ast.literal_eval(to_file.read_text(encoding='utf-8'))
        from_map = {addr: idx for idx, addr in from_list}
        to_map = {addr: idx for idx, addr in to_list}
        return from_map, to_map

    df = pd.read_csv(normal_tx)
    df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0.0)
    df = df[df['value'] != 0.0].copy()
    df['from'] = df['from'].astype(str)
    df['to'] = df['to'].astype(str)

    from_addrs = pd.unique(df['from'])
    to_addrs = pd.unique(df['to'])
    from_list = list(enumerate(from_addrs.tolist()))
    to_list = list(enumerate(to_addrs.tolist()))
    from_file.write_text(str(from_list), encoding='utf-8')
    to_file.write_text(str(to_list), encoding='utf-8')
    from_map = {addr: idx for idx, addr in from_list}
    to_map = {addr: idx for idx, addr in to_list}
    return from_map, to_map

def ensure_holo_tx(case, normal_tx, from_map, to_map):
    case_dir = base / case
    holo_file = case_dir / 'all-normal-tx_holo.csv'
    if holo_file.exists():
        return

    print(f'[{case}] building all-normal-tx_holo.csv from all-normal-tx.csv')
    df = pd.read_csv(normal_tx)
    df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0.0)
    df = df[df['value'] != 0.0].copy()
    df['timeStamp'] = pd.to_numeric(df['timeStamp'], errors='coerce').fillna(0).astype(int)
    df['from'] = df['from'].astype(str)
    df['to'] = df['to'].astype(str)
    df = df[df['from'].isin(from_map) & df['to'].isin(to_map)]

    dates = pd.to_datetime(df['timeStamp'], unit='s', utc=True, errors='coerce').dt.strftime('%Y-%m-%d')
    holo_df = pd.DataFrame({
        0: df['from'].map(from_map).astype('int64'),
        1: df['to'].map(to_map).astype('int64'),
        2: dates,
        3: df['value'].astype(float),
        4: 1,
    }).dropna(subset=[2])
    holo_df.to_csv(holo_file)

for casename in cases_now:
    print(f'Preparing {casename} ...')
    normal_tx = ensure_normal_tx(casename)
    from_a2n, to_a2n = ensure_addr_maps(casename, normal_tx)
    ensure_holo_tx(casename, normal_tx, from_a2n, to_a2n)
    print(f'[{casename}] preparation complete')

Preparing AlphaHomora ...
[AlphaHomora] preparation complete
Preparing CryptopiaHacker ...
[CryptopiaHacker] preparation complete
Preparing PlusTokenPonzi ...
[PlusTokenPonzi] preparation complete


In [9]:
CASES

['AlphaHomora',
 'AscendEXHacker',
 'FakeMetadiumPresale',
 'KucoinHacker',
 'PlusTokenPonzi',
 'UpbitHack']

In [10]:
cases_now = ['AlphaHomora', 
             'CryptopiaHacker', 
             'PlusTokenPonzi'
             ]
for casename in cases_now:
    print('Now processing case',casename,'data preprocessing...\n')
    print('Reading data')
    address = pd.read_csv(f'./inputData/AML/'+casename+'/all-normal-address.csv')
    heist = address.loc[address['label']=='heist']
    heist_address = heist['address'].tolist()
    from_a2n,to_a2n =  holo.addr_to_num(casename)
    from_n2a,to_n2a = holo.num_to_addr(casename)
    filePath =  f'./inputData/AML/'+casename+'/all-normal-tx_holo.csv'
    print('Processing data')
    tensor_data = st.loadTensor(path = filePath, header=None)
    # Format
    tensor_data.data = tensor_data.data.drop(columns=[0])
    tensor_data.data = tensor_data.data.drop([0])
    tensor_data.data.columns=[0,1,2,3,4]
    tensor_data.data[2] = tensor_data.data[2].astype(str).str.replace('/','-')
    tensor_data.data[[0,1,3,4]] = tensor_data.data[[0,1,3,4]].apply(pd.to_numeric, errors='coerce')
    tensor_data.data[2] = pd.to_datetime(tensor_data.data[2], errors='coerce')
    tensor_data.data = tensor_data.data.dropna(subset=[0,1,2,3,4])
    tensor_data.data[3] = pd.cut(tensor_data.data[3], bins=5, labels=[1,2,3,4,5], include_lowest=True).astype('int64')
    tensor_data.data[2] = pd.factorize(tensor_data.data[2])[0]
    tensor_data.data[[0,1,2,3,4]] = tensor_data.data[[0,1,2,3,4]].astype('int64')
    stensor = tensor_data.toSTensor(hasvalue=True)
    graph = st.Graph(stensor, bipartite=True, weighted=True, modet=2)
    hs = st.HoloScope(graph)
    print(casename,'Data construction completed')
    for k in [10, 30]:
        print('Now k=',k,'calculating level=3 results')
        try:
            res3 = hs.run(level=3, k=k)
            add_h3 = outputheist(res3, from_n2a, to_n2a)
            a3 = pd.DataFrame(add_h3, columns=['heist3'])
            out_xlsx = f'./inputData/AML/{casename}/myheist_k_{k}.xlsx'
            with pd.ExcelWriter(out_xlsx) as writer:
                pd.DataFrame(columns=['heist0']).to_excel(writer, 'Sheet0')
                pd.DataFrame(columns=['heist1']).to_excel(writer, 'Sheet1')
                pd.DataFrame(columns=['heist2']).to_excel(writer, 'Sheet2')
                a3.to_excel(writer, 'Sheet3')
            print('Completed',casename,' k=',k,' level=3 result calculation and storage')
        except Exception as e:
            print('Failed',casename,' k=',k,' level=3, error=',e)


Now processing case AlphaHomora data preprocessing...

Reading data
Processing data
AlphaHomora Data construction completed
Now k= 10 calculating level=3 results
initial...
Considering +[topology] +[timestamps] +[sudden drop]+[rating i.e. # of stars] 
matrix size: 29696 x 7690	#edges: 83610.0
finished computing weight matrix
alg: fastgreedy
	+ # of singlular vectors: 10

initial start
Generate tensorfile with time rescale:  1
::::matricize time cost:  0.28504529999997885
matricize 29696x47766 and svd dense... ...
::::Finish Init @  0.3808687000000077
fast greedy algorithm ...
process 1-th singular vector
*** *** shaving ...
set up the greedy min tree
.
*** *** shaving opt size: 1
*** *** shaving opt value: 12.571826258420984
=== === improved opt size: 1
=== === improved opt value: 12.571826258420984
process 2-th singular vector
*** *** shaving ...
set up the greedy min tree
.
*** *** shaving opt size: 1
*** *** shaving opt value: 29.31361490464351
=== === improved opt size: 1
=== === i

c:\Users\sussi\OneDrive\Documents\Denseflow\DenseFlow_Tool\.venv_py38\lib\site-packages\scipy\stats\_continuous_distns.py:6932: RuntimeWarning: divide by zero encountered in scalar divide
  return ndata / np.sum(np.log((data - location) / scale))
c:\Users\sussi\OneDrive\Documents\Denseflow\DenseFlow_Tool\.venv_py38\lib\site-packages\scipy\stats\_continuous_distns.py:6955: RuntimeWarning: invalid value encountered in scalar subtract
  return dL_dLocation(shape, location) - dL_dScale(shape, scale)


Completed AlphaHomora  k= 10  level=3 result calculation and storage
Now k= 30 calculating level=3 results
initial...
Considering +[topology] +[timestamps] +[sudden drop]+[rating i.e. # of stars] 
matrix size: 29696 x 7690	#edges: 83610.0
finished computing weight matrix
alg: fastgreedy
	+ # of singlular vectors: 10

initial start
Generate tensorfile with time rescale:  1
::::matricize time cost:  0.3927888999999709
matricize 29696x47766 and svd dense... ...
::::Finish Init @  0.4852453999999966
fast greedy algorithm ...
process 1-th singular vector
*** *** shaving ...
set up the greedy min tree
.
*** *** shaving opt size: 1
*** *** shaving opt value: 12.571826258420984
=== === improved opt size: 1
=== === improved opt value: 12.571826258420984
process 2-th singular vector
*** *** shaving ...
set up the greedy min tree
.
*** *** shaving opt size: 1
*** *** shaving opt value: 29.31361490464351
=== === improved opt size: 1
=== === improved opt value: 29.31361490464351
process 3-th singul

c:\Users\sussi\OneDrive\Documents\Denseflow\DenseFlow_Tool\.venv_py38\lib\site-packages\scipy\stats\_continuous_distns.py:6932: RuntimeWarning: divide by zero encountered in scalar divide
  return ndata / np.sum(np.log((data - location) / scale))
c:\Users\sussi\OneDrive\Documents\Denseflow\DenseFlow_Tool\.venv_py38\lib\site-packages\scipy\stats\_continuous_distns.py:6955: RuntimeWarning: invalid value encountered in scalar subtract
  return dL_dLocation(shape, location) - dL_dScale(shape, scale)


Completed AlphaHomora  k= 30  level=3 result calculation and storage
Now processing case CryptopiaHacker data preprocessing...

Reading data
Processing data
CryptopiaHacker Data construction completed
Now k= 10 calculating level=3 results
initial...
Considering +[topology] +[timestamps] +[sudden drop]+[rating i.e. # of stars] 
matrix size: 35528 x 82581	#edges: 315169.0
finished computing weight matrix
alg: fastgreedy
	+ # of singlular vectors: 10

initial start
Generate tensorfile with time rescale:  1
::::matricize time cost:  1.1394881999999598
matricize 35528x170269 and svd dense... ...
::::Finish Init @  1.2665848000000324
fast greedy algorithm ...
process 1-th singular vector
*** *** shaving ...
set up the greedy min tree
.
*** *** shaving opt size: 1
*** *** shaving opt value: 34.42924114267665
=== === improved opt size: 1
=== === improved opt value: 34.42924114267665
process 2-th singular vector
*** *** shaving ...
set up the greedy min tree
.
*** *** shaving opt size: 1
*** **

...


	Row and nonzero columns 1 x 22, edge size: 22 with camouflage.
block10: 
	objective value 35.55910006115707
	Node size: 1 x 1, edge size 1
	Row and nonzero columns 1 x 9, edge size: 9 with camouflage.
block11: 
	objective value 32.59363878055927
	Node size: 1 x 1, edge size 1
	Row and nonzero columns 1 x 1, edge size: 1 with camouflage.
block12: 
	objective value 30.77755169763601
	Node size: 1 x 1, edge size 1
	Row and nonzero columns 1 x 1, edge size: 1 with camouflage.
block13: 
	objective value 17.15283872136018
	Node size: 1 x 1, edge size 1
	Row and nonzero columns 1 x 6, edge size: 6 with camouflage.
block14: 
	objective value 12.750121226177518
	Node size: 1 x 1, edge size 1
	Row and nonzero columns 1 x 21, edge size: 21 with camouflage.
block15: 
	objective value 11.066790049367357
	Node size: 1 x 3, edge size 3
	Row and nonzero columns 1 x 349, edge size: 349 with camouflage.
block16: 
	objective value 31.170376444368316
	Node size: 1 x 1, edge size 1
	Row and nonzero colum

c:\Users\sussi\OneDrive\Documents\Denseflow\DenseFlow_Tool\.venv_py38\lib\site-packages\scipy\stats\_continuous_distns.py:6932: RuntimeWarning: divide by zero encountered in scalar divide
  return ndata / np.sum(np.log((data - location) / scale))
c:\Users\sussi\OneDrive\Documents\Denseflow\DenseFlow_Tool\.venv_py38\lib\site-packages\scipy\stats\_continuous_distns.py:6955: RuntimeWarning: invalid value encountered in scalar subtract
  return dL_dLocation(shape, location) - dL_dScale(shape, scale)


finished computing weight matrix
alg: fastgreedy
	+ # of singlular vectors: 10

initial start
Generate tensorfile with time rescale:  1
::::matricize time cost:  0.33027229999993324
matricize 29529x39517 and svd dense... ...
::::Finish Init @  0.41492679999998927
fast greedy algorithm ...
process 1-th singular vector
*** *** shaving ...
set up the greedy min tree
....
*** *** shaving opt size: 4
*** *** shaving opt value: 4.022239608827926
=== === improved opt size: 4
=== === improved opt value: 4.022239608827926
process 2-th singular vector
*** *** shaving ...
set up the greedy min tree
.
*** *** shaving opt size: 1
*** *** shaving opt value: 1.3669003723665814
process 3-th singular vector
*** *** shaving ...
set up the greedy min tree
..
*** *** shaving opt size: 2
*** *** shaving opt value: 2.708935584020059
process 4-th singular vector
*** *** shaving ...
set up the greedy min tree
...
*** *** shaving opt size: 3
*** *** shaving opt value: 0.9141095445163271
process 5-th singular v

c:\Users\sussi\OneDrive\Documents\Denseflow\DenseFlow_Tool\.venv_py38\lib\site-packages\scipy\stats\_continuous_distns.py:6932: RuntimeWarning: divide by zero encountered in scalar divide
  return ndata / np.sum(np.log((data - location) / scale))
c:\Users\sussi\OneDrive\Documents\Denseflow\DenseFlow_Tool\.venv_py38\lib\site-packages\scipy\stats\_continuous_distns.py:6955: RuntimeWarning: invalid value encountered in scalar subtract
  return dL_dLocation(shape, location) - dL_dScale(shape, scale)



	Row and nonzero columns 8 x 6, edge size: 14 with camouflage.
block23: 
	objective value 0.030761775894791763
	Node size: 11 x 1, edge size 5
	Row and nonzero columns 11 x 6, edge size: 16 with camouflage.
block24: 
	objective value 0.05375774723376626
	Node size: 19 x 0, edge size 0
	Row and nonzero columns 19 x 92, edge size: 135 with camouflage.
block25: 
	objective value 0.0014365745860973826
	Node size: 4 x 1, edge size 4
	Row and nonzero columns 4 x 3, edge size: 8 with camouflage.
block26: 
	objective value 0.04629892062749751
	Node size: 3 x 1, edge size 1
	Row and nonzero columns 3 x 3, edge size: 5 with camouflage.
block27: 
	objective value 0.05823601982349506
	Node size: 4 x 1, edge size 3
	Row and nonzero columns 4 x 2, edge size: 5 with camouflage.
block28: 
	objective value 0.0008254836998038033
	Node size: 1 x 1, edge size 1
	Row and nonzero columns 1 x 1, edge size: 1 with camouflage.
block29: 
	objective value 0.03315435612732929
	Node size: 5 x 1, edge size 3
	Row 

In [11]:
caseing = ['AlphaHomora', 'CryptopiaHacker', 'PlusTokenPonzi']
for casename in caseing:
    gpath = './Maxflow_graph/'+casename+'/'
    already = os.path.exists(gpath+'start_nodes.npy')
    if not already:
        print('Constructing graph')
        start_nodes,end_nodes,capacities,nodenum = build_g(casename)
    else:
        print('Reading')
        start_nodes,end_nodes,capacities,nodenum = read_g(casename)
    sourceadds = saddset[casename]
    if sourceadds[0] not in nodenum:
        print('Source address not in graph, skipping case:', casename)
        continue
    for k in [10, 30]:
        level = 3
        print('Calculating k=',k,' level=',level,'ing')
        r=output_flow_add(casename,k,level,sourceadds[0])
    print('Completed')


Constructing graph
构造图ing
完成构建图，节点数： 29696 边数： 83441
new folder
Calculating k= 10  level= 3 ing
当前正在运行： AlphaHomora k= 10 level= 3
读取对应heist地址...
已完成构建，读取图ing
读取完成
进行最大流算法...
共 0 条非0流，总流值： 0
地址存储中...
new folder
Calculating k= 30  level= 3 ing
当前正在运行： AlphaHomora k= 30 level= 3
读取对应heist地址...
已完成构建，读取图ing
读取完成
进行最大流算法...
共 0 条非0流，总流值： 0
地址存储中...
There is this folder
Completed
Constructing graph
构造图ing
完成构建图，节点数： 84862 边数： 315163
new folder
Calculating k= 10  level= 3 ing
当前正在运行： CryptopiaHacker k= 10 level= 3
读取对应heist地址...
已完成构建，读取图ing
读取完成
进行最大流算法...
共 0 条非0流，总流值： 0
地址存储中...
new folder
Calculating k= 30  level= 3 ing
当前正在运行： CryptopiaHacker k= 30 level= 3
读取对应heist地址...
已完成构建，读取图ing
读取完成
进行最大流算法...
共 0 条非0流，总流值： 0
地址存储中...
There is this folder
Completed
Reading
已完成构建，读取图ing
读取完成
Calculating k= 10  level= 3 ing
当前正在运行： PlusTokenPonzi k= 10 level= 3
读取对应heist地址...
已完成构建，读取图ing
读取完成
进行最大流算法...
共 174 条非0流，总流值： 2334662
地址存储中...
There is this folder
Calculating k= 30  level= 3 ing
当前正在运行： P

In [12]:
g = os.walk(f'./Result/')  
cases = []
for path,dir_list,file_list in g:  
    for dir_name in dir_list:  
        cases.append(str(dir_name[0:-4]))
        print(dir_name[0:-4])
TUANHUOJIANCE(cases,k)

AlphaHomora
CryptopiaHacker
PlusTokenPonzi
缺少结果文件 AlphaHomora_k_30_level_0
缺少结果文件 AlphaHomora_k_30_level_1
缺少结果文件 AlphaHomora_k_30_level_2
缺少结果文件 AlphaHomora_k_30_level_3
缺少结果文件 CryptopiaHacker_k_30_level_0
缺少结果文件 CryptopiaHacker_k_30_level_1
缺少结果文件 CryptopiaHacker_k_30_level_2
缺少结果文件 CryptopiaHacker_k_30_level_3
缺少结果文件 PlusTokenPonzi_k_30_level_0
缺少结果文件 PlusTokenPonzi_k_30_level_1
缺少结果文件 PlusTokenPonzi_k_30_level_2
缺少结果文件 PlusTokenPonzi_k_30_level_3


,precision_level_0,precision_level_1,precision_level_2,precision_level_3,recall_level_0,recall_level_1,recall_level_2,recall_level_3
AlphaHomora,-1,-1,-1,-1,-1,-1,-1,-1
CryptopiaHacker,-1,-1,-1,-1,-1,-1,-1,-1
PlusTokenPonzi,-1,-1,-1,-1,-1,-1,-1,-1


In [13]:
import pandas as pd

casename = ['AlphaHomora', 'CryptopiaHacker', 'PlusTokenPonzi']
k_values = [10, 30]
level = 3

rows = []
for casename in cases:
    for k in k_values:
        metrics = check_metrics_my(casename, level=level, k=k)
        pre, mcr, msize = metrics['HOLO+MAXFLOW']
        rows.append({
            'case': casename,
            'k': k,
            'level': level,
            'precision': pre,
            'mcr': mcr,
            '|M|': msize,
        })

metrics_df = pd.DataFrame(rows)
display(metrics_df)

#out_csv = './Result/metrics_level_3_k_10_30.csv'
#out_xlsx = './Result/metrics_level_3_k_10_30.xlsx'
#metrics_df.to_csv(out_csv, index=False)
#with pd.ExcelWriter(out_xlsx) as writer:
#    metrics_df.to_excel(writer, index=False)


HOLO+MAXFLOW (level=3 k=10)
总共找出： 19 找出正确的：(heist) 0 ，
正确率(Precision): 0.0 ,
召回率（Recall）： 0.0 


追踪金额： 1.247272562711582e-16 涉案金额： 3.961512135163169e-14 资金覆盖率: 0.0031484759358441513
账户总数： 29696 检测heist： 19

HOLO+MAXFLOW (level=3 k=30)
总共找出： 49 找出正确的：(heist) 6 ，
正确率(Precision): 0.12244897959183673 ,
召回率（Recall）： 0.0008620689655172414 


追踪金额： 1.76184446534355e-16 涉案金额： 3.961512135163169e-14 资金覆盖率: 0.004447403933727902
账户总数： 29696 检测heist： 49

HOLO+MAXFLOW (level=3 k=10)
总共找出： 20 找出正确的：(heist) 2 ，
正确率(Precision): 0.1 ,
召回率（Recall）： 0.00022760896779333106 


追踪金额： 1.5428956000978722e-13 涉案金额： 3.6410920922556475e-13 资金覆盖率: 0.4237452832845138
账户总数： 84862 检测heist： 20

HOLO+MAXFLOW (level=3 k=30)
总共找出： 74 找出正确的：(heist) 5 ，
正确率(Precision): 0.06756756756756757 ,
召回率（Recall）： 0.0005690224194833276 


追踪金额： 5.806772232827564e-13 涉案金额： 3.6410920922556475e-13 资金覆盖率: 1
账户总数： 84862 检测heist： 74

HOLO+MAXFLOW (level=3 k=10)
总共找出： 19559 找出正确的：(heist) 19364 ，
正确率(Precision): 0.9900301651413671 ,
召回率（Rec

,case,k,level,precision,mcr,|M|
0,AlphaHomora,10,3,0.000000,0.003148,19
1,AlphaHomora,30,3,0.122449,0.004447,49
2,CryptopiaHacker,10,3,0.100000,0.423745,20
3,CryptopiaHacker,30,3,0.067568,1.594789,74
4,PlusTokenPonzi,10,3,0.990030,0.831389,19559
5,PlusTokenPonzi,30,3,0.983149,0.835178,19940
